AI_StudyBuddy


In [28]:
import os
from dotenv import load_dotenv
load_dotenv()


True

Defining State


In [31]:
from typing import TypedDict
from typing import Annotated
from langgraph.graph.message import add_messages
class State(TypedDict):
    messages: Annotated[list, add_messages]

Creating LLM

In [18]:
from langchain.chat_models import init_chat_model
llm=init_chat_model("gpt-4o-mini",model_provider="openAI")

Defining Tools

Calculator


In [19]:
def calculator(task:str,a:int,b:int)->float:
    """Accepts 3 parameter to perform mathematical operations 
    Arg1 task : str 'add' adds two number 'subtract' subtracts two number 'multiply' multiplies two numbers 'divide' divides two number 'average' average of two numbers
    Arg2 a integer number
    Arg3 integer number
    
    returns the operated result in float"""

    if task=="add":
        return a+b
    elif task=="subtract":
        return a-b
    elif task=="multiply":
        return a*b
    elif task=="divide":
        return a/b
    else:
        return (a+b)/2

    

Binding tools


In [20]:
tools=[calculator]
llm_with_tools=llm.bind_tools(tools)

Writing Node Definition

In [33]:
def ai_StudyBuddy(state: State):
    return {"messages": [llm_with_tools.invoke(state["messages"])]}

Building GraphStructure

In [38]:
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode,tools_condition



graph=StateGraph(State)
graph.add_node("aiStudyBuddy",ai_StudyBuddy)
graph.add_node("tools",ToolNode(tools))
graph.add_edge(START,"aiStudyBuddy")
graph.add_conditional_edges("aiStudyBuddy",tools_condition)
graph.add_edge("tools", "aiStudyBuddy")


ai_StudyBuddy_APP=graph.compile()

response=ai_StudyBuddy_APP.invoke({"messages":"Add the two numbers 2 and 5"})

response["messages"][-1].content




'The sum of the numbers 2 and 5 is 7.'